# Phase 3 — PPO-V3 Coordination & True Stabilization Training 🔥

This notebook trains PPO-V3 models on Google Colab with:
- **Fixed initialization starting positions** (no overlaps for team sizes 1–10)
- **Leakage-free agent suppression tracking** (ignores natural fire decay)
- **Massively reduced overlap penalty** (`-0.05` instead of `-2.0`)
- **Balanced reward scales** (Coverage weight `2.0`, active suppression weight `5.0`)
- **Extended MARL budgets** (3 agents: 300k, 5 agents: 500k, 10 agents: 1M steps)

### Instructions:
1. Compress your workspace (exclude `models/` to keep it small) and name it `wildfire-rl.zip`.
2. Upload it to Colab's file pane.
3. Set runtime to **T4 GPU**.
4. Run all cells sequentially.

### 1. Extract Workspace Code

In [ ]:
import os
from pathlib import Path

zip_name = "wildfire-rl.zip"
if Path(zip_name).exists():
    !unzip -q {zip_name} -d wildfire-rl
    %cd wildfire-rl
    print(f"Entered workspace: {os.getcwd()}")
else:
    print(f"ERROR: upload '{zip_name}' first.")

### 2. Install Project Dependencies

In [ ]:
!grep -v "torch" requirements.txt | grep -v "numpy" > req_clean.txt
!pip install -q -r req_clean.txt
!pip install -q -e .

### 3. Verify GPU Availability

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU is not active. Go to Runtime -> Change runtime type and select T4 GPU.")

### 4. Run Parallel Subprocess PPO MARL V3 Training on GPU (Matched Budgets)

In [ ]:
import subprocess
import time
from pathlib import Path

FORCE_RETRAIN = True
MAX_PARALLEL = 4

# Extended budget mappings per team size
TIMESTEPS_MAP = {
    3: 300000,
    5: 500000,
    10: 1000000
}

jobs = []
for region in ["saudi", "california"]:
    for n_agents, timesteps in TIMESTEPS_MAP.items():
        for seed in [0, 1, 2]:
            jobs.append((region, n_agents, seed, timesteps))

processes = []
print(f"Launching {len(jobs)} V3 training runs in parallel using {MAX_PARALLEL} processes...")

for region, agents, seed, timesteps in jobs:
    model_file = Path("models") / f"ppo_v3_marl_{region}_{agents}agents_seed_{seed}.zip"
    if model_file.exists() and not FORCE_RETRAIN:
        print(f"Skip: {model_file.name} already exists.")
        continue
        
    # Limit parallel execution
    while len(processes) >= MAX_PARALLEL:
        for p in list(processes):
            if p.poll() is not None:
                processes.remove(p)
        time.sleep(1)

    cmd = [
        "python", "scripts/train_single_marl_v3.py",
        "--region", region,
        "--agents", str(agents),
        "--seed", str(seed),
        "--timesteps", str(timesteps)
    ]
    print(f"Launching: {region}, {agents} agents, {timesteps} steps, seed={seed}")
    p = subprocess.Popen(cmd)
    processes.append(p)

for p in processes:
    p.wait()

print("\nAll parallel training runs finished successfully!")

### 5. Run V3 Scientific Evaluation Pipeline

In [ ]:
!python scripts/run_marl_v3_evaluation.py
print("\n✅ V3 evaluation complete. Check results/v3/ and figures/v3/ for outputs.")

### 6. Package and Download Trained Checkpoints & Outputs

In [ ]:
!zip -j v3_models.zip models/ppo_v3_marl_*
!zip -r v3_results.zip results/v3/ figures/v3/

print("\n--- READY FOR DOWNLOAD ---")
print("1. Click the file explorer icon in Colab (left panel).")
print("2. Download 'v3_models.zip' and 'v3_results.zip'.")
print("3. Extract v3_models.zip locally into your 'models/' folder.")
print("4. Extract v3_results.zip into your repo root.")